# Q-Learning for Tic-Tac-Toe

### The RL Framework

RL is built on a simple interaction loop:

1. **Agent**: Our learning algorithm (the tic-tac-toe player)
2. **Environment**: The game world (the tic-tac-toe board)
3. **State (S)**: Current situation (board configuration)
4. **Action (A)**: What the agent can do (place X or O in a cell)
5. **Reward (R)**: Feedback from the environment (+1 for win, -1 for loss, 0 for draw)

**The Loop:**
- Agent observes **State** → chooses **Action** → receives **Reward** & new **State**
- Agent learns from this experience
- Repeat thousands of times until the agent becomes an expert!


<img src="https://gymnasium.farama.org/_images/AE_loop.png" alt="RL Loop" width="360">


## Understanding Q-Learning

**Q-Learning** is like building a massive "cheat sheet" that tells our agent how good each move is in every possible game situation.

### The Q-Table: Agent's Brain

Think of the Q-table as a spreadsheet:
- **Rows**: All possible board configurations (states)
- **Columns**: All possible moves (actions: cells 1-9)
- **Values**: Q-values = "How good is this move in this situation?"

### How Q-Values Work

- **High Q-value** (e.g., +0.8): "This move leads to winning!"
- **Low Q-value** (e.g., -0.5): "This move leads to losing!"
- **Zero Q-value**: "I don't know yet, haven't tried this."

### The Learning Process

1. **Initialize**: Start with Q-table full of zeros (agent knows nothing)
2. **Play games**: Agent tries moves and observes outcomes
3. **Update Q-values**: After each move, update the Q-table based on reward
4. **Improve strategy**: Over time, good moves get higher Q-values

### The Magic Formula: Bellman Equation

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

Let's decode this:

- $Q(s, a)$: Current Q-value for state $s$, action $a$
- $\alpha$ (alpha): **Learning rate** (0-1) = How fast we learn
- $r$: **Reward** we just received
- $\gamma$ (gamma): **Discount factor** (0-1) = How much we value future rewards
- $\max_{a'} Q(s', a')$: Best possible Q-value from next state
- $[...]$: **Temporal Difference (TD) error** = difference between prediction and reality

**In plain English:**

*"Update my estimate of how good this move is, based on the immediate reward I got plus the best possible outcome from where I landed, minus what I previously thought."*

### Exploration vs Exploitation

**The Dilemma:**
- **Exploit**: Use what you know (pick the best move from Q-table)
- **Explore**: Try something new (maybe there's a better move!)

**Solution: ε-Greedy Policy (Epsilon-Greedy)**

- With probability $\epsilon$ (epsilon): **Explore** (random move)
- With probability $1 - \epsilon$: **Exploit** (best known move)

At the start, $\epsilon = 1.0$ (100% exploration). Over time, we reduce it to $\epsilon = 0.01$ (1% exploration), so the agent learns to trust its knowledge but still occasionally tries new things.

## Tic-Tac-Toe as an RL Problem

Let's map tic-tac-toe to the RL framework:

### State Space

**State** = Current board configuration

```
Board positions:        Example state:
 1 | 2 | 3               X | O | 3
-----------             -----------
 4 | 5 | 6               X | 5 | 6
-----------             -----------
 7 | 8 | 9               7 | 8 | 9
```

We represent this as a tuple: `('X', 'O', None, 'X', None, None, None, None, None)`

**Total possible states**: $3^9 = 19,683$ (each cell can be X, O, or empty)
- Many are unreachable (illegal game states)
- Agent will only learn the relevant ones!

### Action Space

**Action** = Choose a cell to place your symbol
- Actions: integers 1-9 (cell positions)
- Only empty cells are valid actions

### Reward Structure

We need to define what's "good" and "bad":

- **Win**: +1.0 (Great job!)
- **Loss**: -1.0 (Learn from this mistake)
- **Draw**: 0.0 (Not bad, but not winning)
- **Each move**: -0.01 (Small penalty to encourage fast wins)

Why penalize each move? So the agent learns to win quickly rather than prolonging the game!

### Winning Conditions

8 ways to win:
- 3 rows: (1,2,3), (4,5,6), (7,8,9)
- 3 columns: (1,4,7), (2,5,8), (3,6,9)
- 2 diagonals: (1,5,9), (3,5,7)

## Step 1: Setup and Imports

In [ ]:
# Import necessary libraries
import numpy as np
import random
import pickle
import matplotlib.pyplot as plt


# Import the environment from the training package
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

## Step 2: The Environment (Tic-Tac-Toe Game)

Now let's import the `TicTacToeEnvironment` class from our training module. This class handles:
- Board state management
- Move validation
- Win condition checking
- Board display

**Note:** We import it from the `training` package which contains all our reusable Q-Learning components.

In [ ]:
from training.environment import TicTacToeEnvironment

# Test the environment
env = TicTacToeEnvironment()
print("Initial board:")
env.display()

# Make some moves
env.make_move(5, 'X')  # Center
env.make_move(1, 'O')  # Top-left
env.make_move(9, 'X')  # Bottom-right
print("After some moves:")
env.display()

print(f"Available actions: {env.get_available_actions()}")
print(f"Winner: {env.check_winner()}")
print(f"Game over: {env.is_game_over()}")

print("\n📄 The TicTacToeEnvironment class is defined in: ../training/environment.py")

## Step 3: The Q-Learning Agent

Now let's import the `QLearningAgent` class. The agent:

1. **Stores knowledge**: Q-table (dictionary mapping state-action pairs to Q-values)
2. **Chooses actions**: ε-greedy policy (explore vs exploit)
3. **Learns from experience**: Updates Q-values using Bellman equation

### Key Components:

- **Q-table**: `{(state, action): Q-value}`
- **Hyperparameters**: α (learning rate), γ (discount), ε (exploration)
- **Episode history**: Tracks states and actions to update after each move

In [ ]:
# Import the Q-Learning agent
from training.q_learning_agent import QLearningAgent

# Create an agent
agent = QLearningAgent(symbol='X', epsilon=0.3)

print("Q-Learning Agent created!")
print(f"\nAgent configuration:")
print(f"  Symbol: {agent.symbol}")
print(f"  Learning rate (α): {agent.learning_rate}")
print(f"  Discount factor (γ): {agent.discount_factor}")
print(f"  Exploration rate (ε): {agent.epsilon}")

print("\n📄 The QLearningAgent class is defined in: ../training/q_learning_agent.py")

## Step 4: Training the Agent

We'll have our agent play thousands of games against a random opponent.

### Training Process:

1. **Play many episodes** (games)
2. **Start with high exploration** (ε = 1.0) - agent tries random moves
3. **Gradually reduce exploration** - agent starts trusting learned Q-values
4. **Update Q-table after each move** using Bellman equation
5. **Track performance** - win rate, loss rate, draw rate

### Hyperparameters:

- **Episodes**: 20,000 games
- **Learning rate (α)**: 0.1 - moderate learning speed
- **Discount factor (γ)**: 0.9 - value future rewards highly
- **Epsilon decay**: 1.0 → 0.01 - start exploring, end exploiting

In [ ]:
# Define training function
def play_training_game(agent: QLearningAgent, env: TicTacToeEnvironment, 
                      agent_first: bool = True) -> str:
    """
    Play one training game: agent vs random opponent.
    
    Args:
        agent: Q-learning agent
        env: Game environment
        agent_first: If True, agent goes first
    
    Returns:
        'win', 'loss', or 'draw' from agent's perspective
    """
    env.reset()
    agent.reset_episode()
    
    agent_symbol = agent.symbol
    opponent_symbol = 'O' if agent_symbol == 'X' else 'X'
    current_player = agent_symbol if agent_first else opponent_symbol
    
    while True:  # Game loop
        available = env.get_available_actions()
        
        if current_player == agent_symbol:  # Agent's turn
            move = agent.choose_action(env.board, available)
        else:  # Random opponent's turn
            move = random.choice(available)
        
        env.make_move(move, current_player)
        is_over, result = env.is_game_over()
        
        if current_player == agent_symbol:
            # Agent just moved - time to learn!
            if is_over:
                # Game ended - learn with final reward and next_board=None
                if result == agent_symbol:
                    reward = 1.0  # WIN!
                elif result == opponent_symbol:
                    reward = -1.0  # LOSS!
                else:
                    reward = 0.0  # DRAW
                
                # Learn with next_board=None to indicate terminal state
                agent.learn(reward=reward, next_board=None)
                agent.reset_episode()  # Clear history for next game
                
                return 'win' if result == agent_symbol else ('loss' if result == opponent_symbol else 'draw')
            else:
                # Game continues - small penalty for each move
                agent.learn(reward=-0.01, next_board=env.board)
        elif is_over:
            # Opponent's move ended the game
            agent.reset_episode()  # Clear history for next game
            return 'loss' if result == opponent_symbol else 'draw'
        
        # Switch player
        current_player = opponent_symbol if current_player == agent_symbol else agent_symbol

print("✓ Training function defined!")

In [ ]:
# Training parameters
N_EPISODES = 20000
EVAL_EVERY = 2000  # Evaluate every N episodes

# Epsilon decay parameters
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 0.9995

# Create fresh agent and environment
agent = QLearningAgent(
    symbol='X',
    learning_rate=0.1,
    discount_factor=0.9,
    epsilon=EPSILON_START,
    training_mode=True
)
env = TicTacToeEnvironment()

print("=" * 60)
print("  Training Q-Learning Agent for Tic-Tac-Toe")
print("=" * 60)
print(f"Episodes: {N_EPISODES}")
print(f"Learning rate (α): {agent.learning_rate}")
print(f"Discount factor (γ): {agent.discount_factor}")
print(f"Epsilon: {EPSILON_START} → {EPSILON_END}")
print("\nTraining in progress...\n")

# Statistics tracking for visualization
stats = {
    'episodes': [],
    'win_rate': [],
    'loss_rate': [],
    'draw_rate': [],
    'epsilon': [],
    'q_table_size': []
}

# Training statistics
wins = losses = draws = 0
epsilon = EPSILON_START

for episode in range(1, N_EPISODES + 1):
    # Decay epsilon (reduce exploration over time)
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    agent.epsilon = epsilon
    
    # Alternate who goes first
    agent_first = random.choice([True, False])
    
    # Play one game
    result = play_training_game(agent, env, agent_first)
    
    # Update stats
    if result == 'win':
        wins += 1
    elif result == 'loss':
        losses += 1
    else:
        draws += 1
    
    # Record statistics for plotting
    if episode % EVAL_EVERY == 0:
        total = wins + losses + draws
        stats['episodes'].append(episode)
        stats['win_rate'].append(wins / total * 100)
        stats['loss_rate'].append(losses / total * 100)
        stats['draw_rate'].append(draws / total * 100)
        stats['epsilon'].append(epsilon)
        stats['q_table_size'].append(len(agent.q_table))
        
        # Reset counters
        wins = losses = draws = 0

print("=" * 60)
print("Training Complete!")
print("=" * 60)
print(f"Final Q-table size: {len(agent.q_table)} entries")
print(f"Unique states learned: {len(set(s for s, a in agent.q_table.keys()))}")
print()

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Q-Learning Training Progress', fontsize=16, fontweight='bold')

# Plot 1: Win/Loss/Draw rates over time
ax1 = axes[0, 0]
ax1.plot(stats['episodes'], stats['win_rate'], 'g-', linewidth=2, label='Win Rate')
ax1.plot(stats['episodes'], stats['loss_rate'], 'r-', linewidth=2, label='Loss Rate')
ax1.plot(stats['episodes'], stats['draw_rate'], 'b-', linewidth=2, label='Draw Rate')
ax1.set_xlabel('Episode', fontsize=11)
ax1.set_ylabel('Rate (%)', fontsize=11)
ax1.set_title('Win/Loss/Draw Rates', fontsize=12, fontweight='bold')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, N_EPISODES)
ax1.set_ylim(0, 100)

# Plot 2: Epsilon decay
ax2 = axes[0, 1]
ax2.plot(stats['episodes'], stats['epsilon'], 'purple', linewidth=2)
ax2.set_xlabel('Episode', fontsize=11)
ax2.set_ylabel('Epsilon (ε)', fontsize=11)
ax2.set_title('Exploration Rate Decay', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, N_EPISODES)
ax2.set_ylim(0, 1.1)

# Plot 3: Q-table growth
ax3 = axes[1, 0]
ax3.plot(stats['episodes'], stats['q_table_size'], 'orange', linewidth=2)
ax3.set_xlabel('Episode', fontsize=11)
ax3.set_ylabel('Number of Entries', fontsize=11)
ax3.set_title('Q-Table Size Growth', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, N_EPISODES)

# Plot 4: Performance summary (final checkpoint)
ax4 = axes[1, 1]
final_stats = [stats['win_rate'][-1], stats['loss_rate'][-1], stats['draw_rate'][-1]]
colors = ['green', 'red', 'blue']
bars = ax4.bar(['Wins', 'Losses', 'Draws'], final_stats, color=colors, alpha=0.7, edgecolor='black')
ax4.set_ylabel('Rate (%)', fontsize=11)
ax4.set_title('Final Performance Summary', fontsize=12, fontweight='bold')
ax4.set_ylim(0, 100)
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1f}%',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("  • Win rate increases as agent learns optimal strategy")
print("  • Loss rate decreases as agent avoids bad moves")
print("  • Epsilon decays from exploration to exploitation")
print("  • Q-table grows as agent discovers new states")

## Step 5: Evaluation

Training is done! But how good is our agent really?

Let's test it properly:
- Turn OFF exploration (ε = 0)
- Play 1000 games using only learned Q-values
- Calculate win/loss/draw rates

**What to expect:**
- Against random opponent, optimal play should:
  - **Win**: ~90-95% (when going first or opponent makes mistakes)
  - **Draw**: ~5-10% (when opponent accidentally plays perfectly)
  - **Loss**: ~0-1% (should be nearly impossible with optimal play)

In [ ]:
print("Evaluating agent (1000 games, no exploration)...\n")

# Switch to evaluation mode
agent.training_mode = False
agent.epsilon = 0.0

# Play evaluation games
wins = losses = draws = 0

for _ in range(1000):
    agent_first = random.choice([True, False])
    result = play_training_game(agent, env, agent_first)
    
    if result == 'win':
        wins += 1
    elif result == 'loss':
        losses += 1
    else:
        draws += 1

print("=" * 60)
print("Evaluation Results (vs Random Opponent)")
print("=" * 60)
print(f"Wins:   {wins:4d} / 1000  ({wins/10:.1f}%)")
print(f"Losses: {losses:4d} / 1000  ({losses/10:.1f}%)")
print(f"Draws:  {draws:4d} / 1000  ({draws/10:.1f}%)")
print()

if losses < 10:
    print("Excellent! Agent learned near-optimal strategy!")
elif losses < 50:
    print("Good! Agent performs well but could improve.")
else:
    print("Agent needs more training.")

## Step 6: Visualizing Q-Values

Let's peek inside the agent's brain! We'll look at Q-values for a specific game state to understand what the agent learned.

We'll examine an early game position and see which moves the agent thinks are best.

In [ ]:
# Create a more interesting late-game state
# X has two in the top row (1 and 2), can win with 3
# O has blocked center and bottom-left corner
# This should show clear Q-value differences
sample_board = {
    1: 'X', 2: 'X', 3: None,    # X can win here!
    4: None, 5: 'O', 6: None,
    7: 'O', 8: None, 9: None    # O has corner
}

print("Sample board state (X's turn):")
print(" X | X | 3")
print("-----------")
print(" 4 | O | 6")
print("-----------")
print(" O | 8 | 9")
print()

# Get Q-values for all available actions
state = agent.get_state_key(sample_board)
available = [i for i in range(1, 10) if sample_board.get(i) is None]

print("Q-values for available moves (X's turn):")
print()

q_values = {}
for action in available:
    q_val = agent.get_q_value(state, action)
    q_values[action] = q_val
    print(f"  Cell {action}: {q_val:+.3f}")

best_action = max(q_values, key=q_values.get)
print()
print(f"Agent would choose: Cell {best_action} (Q-value: {q_values[best_action]:+.3f})")
print()
print("Interpretation:")
print("  • Cell 3 should have the HIGHEST Q-value → wins immediately!")
print("  • Positive Q-values → moves that likely lead to winning")
print("  • Negative Q-values → moves that likely lead to losing")
print("  • The large difference shows the agent learned to recognize winning moves!")
print("\n💡 The agent's get_q_value() method uses the Q-table learned during training")

## Step 7: Analyzing Learned Strategy

Let's analyze some specific game situations to understand what the agent learned. We'll look at critical board positions and see which moves the agent prefers.

### Why Didn't the Agent Learn Perfect Play?

You might have noticed that the agent **didn't learn to block** in Test 2, and the evaluation results showed ~16% losses against a random opponent. This seems surprising - shouldn't Q-learning converge to optimal play?

#### The Root Cause: Training Against a Weak Opponent

The agent trained exclusively against a **random opponent**, which creates a fundamental problem:

**The agent only learns what it experiences.**

Let's analyze what happens:

1. **Random opponents rarely exploit weaknesses**
   - A random opponent might not take the winning move even when it's available
   - The agent doesn't consistently experience punishment for mistakes
   - Many bad positions never lead to losses during training

2. **Blocking becomes unnecessary**
   - When opponent has two-in-a-row, a random player only wins 1/N times (where N = available moves)
   - Example: If there are 6 empty cells, opponent only wins ~16.7% of the time
   - Other moves might still win often enough to get positive Q-values
   - **The agent learns: "I don't always need to block"**

3. **Incomplete state space exploration**
   - Some critical defensive positions are rarely encountered
   - When they are encountered, losses aren't consistent enough to update Q-values properly
   - The agent develops "blind spots" in its strategy


## Step 8: Save and Load the Q-Table

Training takes time! Let's save our Q-table so we can use it later without retraining.

In [ ]:
# Save Q-table to the training folder
filename = '../training/q_table.pkl'

with open(filename, 'wb') as f:
    pickle.dump(agent.q_table, f)

print(f"Q-table saved to {filename}")
print(f"Size: {len(agent.q_table)} entries")
print()
print("This Q-table can be loaded by the standalone Python scripts")
print("for interactive gameplay:")
print()
print("  04_qlearning/standalone/ttt_qlearning.py  - Simple standalone version")
print("  04_qlearning/training/train_qlearning.py  - Full training script")
print()

# Demonstrate how to load the Q-table
print("To load the Q-table in your own code:")
print()
print("  import pickle")
print("  with open('../training/q_table.pkl', 'rb') as f:")
print("      q_table = pickle.load(f)")
print("      agent.q_table = q_table")
print()
print("Or use the agent's built-in method:")
print("  agent.load('../training/q_table.pkl')")

## Summary

In this notebook, you learned how Q-Learning works and saw it in action for tic-tac-toe!

### Key Takeaways:

1. **Modular Code Organization**: The implementation uses well-organized modules:
   - `environment.py`: Game environment (board, rules, win detection)
   - `q_learning_agent.py`: Q-Learning agent (Q-table, learning, action selection)
   - `train_qlearning.py`: Training pipeline and evaluation

2. **Q-Learning Components**:
   - **Q-Table**: Maps (state, action) → Q-value (expected future reward)
   - **Bellman Equation**: Updates Q-values based on rewards and future estimates
   - **ε-Greedy Policy**: Balances exploration (trying new moves) vs exploitation (using learned knowledge)

3. **Training Process**:
   - Agent learns by playing thousands of games
   - Epsilon decay: starts exploring (ε=1.0), ends exploiting (ε=0.01)
   - Q-values converge to represent the quality of each move

### Play Against the Agent:

Run the interactive gameplay script:
```bash
cd 04_qlearning/standalone
python3 ttt_qlearning.py
```

### Train Your Own Agent:

Train from scratch with custom hyperparameters:
```bash
cd 04_qlearning/training  
python3 train_qlearning.py
```

### Limitations & Future Directions:

The agent trained only against a random opponent, so it has some weaknesses:
- Doesn't always block opponent threats
- Can lose to smarter opponents

**Improvements**:
1. Train against better opponents (minimax, self-play)
2. Handle board symmetries to reduce state space
3. Use Deep Q-Networks (DQN) for more complex games

### Code Files:

- **[environment.py](../training/environment.py)**: TicTacToeEnvironment class
- **[q_learning_agent.py](../training/q_learning_agent.py)**: QLearningAgent class  
- **[train_qlearning.py](../training/train_qlearning.py)**: Training pipeline
- **[ttt_qlearning.py](../standalone/ttt_qlearning.py)**: Interactive gameplay